In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
from pathlib import Path
from IPython.display import IFrame

# =============================================================================
# CONFIGURATION
# =============================================================================
TARGET_TEAM_ID = 185651
LOCAL_MATCHES_FILE = Path('../Data/processed/events/master_match_results.csv')
OUTPUT_HTML = "team_network.html"

print("--- MISSION START: NETWORK VISUALIZATION ---")

# 1. Load and Clean Data
df_matches = pd.read_csv(LOCAL_MATCHES_FILE)
df_matches = df_matches.dropna(subset=['Team_A_ID', 'Team_B_ID'])
df_matches['Team_A_ID'] = df_matches['Team_A_ID'].astype(int)
df_matches['Team_B_ID'] = df_matches['Team_B_ID'].astype(int)

# Create a mapping dictionary to translate IDs to human-readable names
id_to_name = pd.concat([
    df_matches[['Team_A_ID', 'Team_A_Name']].rename(columns={'Team_A_ID': 'ID', 'Team_A_Name': 'Name'}),
    df_matches[['Team_B_ID', 'Team_B_Name']].rename(columns={'Team_B_ID': 'ID', 'Team_B_Name': 'Name'})
]).drop_duplicates(subset=['ID']).set_index('ID')['Name'].to_dict()

# 2. Filter for 1st Degree Connections (To prevent massive browser lag)
# Find all matches where the target team played
target_matches = df_matches[
    (df_matches['Team_A_ID'] == TARGET_TEAM_ID) | 
    (df_matches['Team_B_ID'] == TARGET_TEAM_ID)
]

# Identify all direct opponents
opponents = set(target_matches['Team_A_ID']).union(set(target_matches['Team_B_ID']))

# Now get all matches between any of these teams (Target + Opponents)
filtered_matches = df_matches[
    df_matches['Team_A_ID'].isin(opponents) & 
    df_matches['Team_B_ID'].isin(opponents)
]

# 3. Aggregate Edge Weights (Count how many times they played each other)
# Sort the IDs so A vs B is the same as B vs A
edges = filtered_matches.apply(
    lambda x: tuple(sorted([x['Team_A_ID'], x['Team_B_ID']])), axis=1
).value_counts().reset_index()
edges.columns = ['Matchup', 'Weight']

# 4. Build the NetworkX Graph
G = nx.Graph()

for index, row in edges.iterrows():
    team1, team2 = row['Matchup']
    weight = row['Weight']
    
    name1 = id_to_name.get(team1, str(team1))
    name2 = id_to_name.get(team2, str(team2))
    
    # Add nodes with specific colors (Highlight the target team in Red)
    for team_id, team_name in [(team1, name1), (team2, name2)]:
        if team_id not in G:
            color = "#ff4d4d" if team_id == TARGET_TEAM_ID else "#97c2fc"
            size = 25 if team_id == TARGET_TEAM_ID else 15
            G.add_node(team_id, label=team_name, title=team_name, color=color, size=size)
            
    # Add the edge (line thickness based on number of times played)
    G.add_edge(team1, team2, value=weight, title=f"Played {weight} times")

# 5. Render the Graph with PyVis
print(f"Rendering Network: {G.number_of_nodes()} Teams, {G.number_of_edges()} Matchups.")

# Initialize the interactive network physics map
net = Network(height='600px', width='100%', bgcolor='#ffffff', font_color='black', notebook=True)
net.from_nx(G)

# Tweak physics to make it look organic and prevent overlapping
net.repulsion(node_distance=150, central_gravity=-0.1, spring_length=100)

# Generate the HTML file
net.show(OUTPUT_HTML)
print(f"[SUCCESS] Network saved to {OUTPUT_HTML}")

# Display inside Jupyter
IFrame(OUTPUT_HTML, width='100%', height='620px')
